In [ ]:
# ============================================================
# FINAL COLAB CODE: BUILD CLEAN thesis_master.csv
# SCHEMA KEPT:
# id, movieId, imdb_id, title, overview, keywords_clean,
# cast_clean, director_name, genres_clean, original_language,
# release_date, release_year, budget, revenue, roi_raw,
# runtime, popularity, vote_average, vote_count,
# text_overview_only, text_semantic_enriched, text_semantic_full
# ============================================================

# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Imports
import os
import ast
import shutil
import pandas as pd
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================
PROJECT_DIR = "/content/drive/MyDrive/thesis_movie_project"
RAW_DIR = f"{PROJECT_DIR}/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/processed"

LOCAL_DIR = "/content/thesis_movie_project"
LOCAL_RAW_DIR = f"{LOCAL_DIR}/raw"
LOCAL_PROCESSED_DIR = f"{LOCAL_DIR}/processed"

OUTPUT_FILE = "thesis_master_base.csv"

for d in [PROJECT_DIR, RAW_DIR, PROCESSED_DIR, LOCAL_DIR, LOCAL_RAW_DIR, LOCAL_PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

# Required raw files in Google Drive
MOVIES_FILE = f"{RAW_DIR}/movies_metadata.csv"
KEYWORDS_FILE = f"{RAW_DIR}/keywords.csv"
CREDITS_FILE = f"{RAW_DIR}/credits.csv"
LINKS_FILE = f"{RAW_DIR}/links.csv"   # optional but recommended

# Local copies for faster processing
MOVIES_LOCAL = f"{LOCAL_RAW_DIR}/movies_metadata.csv"
KEYWORDS_LOCAL = f"{LOCAL_RAW_DIR}/keywords.csv"
CREDITS_LOCAL = f"{LOCAL_RAW_DIR}/credits.csv"
LINKS_LOCAL = f"{LOCAL_RAW_DIR}/links.csv"

OUTPUT_LOCAL = f"{LOCAL_PROCESSED_DIR}/{OUTPUT_FILE}"
OUTPUT_DRIVE = f"{PROCESSED_DIR}/{OUTPUT_FILE}"

TOP_CAST = 5

# ============================================================
# HELPERS
# ============================================================
def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

def copy_to_local(src, dst):
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

def safe_parse_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if x == "" or x == "[]" or x.lower() == "nan":
        return []
    try:
        val = ast.literal_eval(x)
        return val if isinstance(val, list) else []
    except Exception:
        return []

def extract_names(x, top_n=None):
    items = safe_parse_list(x)
    names = []
    for item in items:
        if isinstance(item, dict):
            name = str(item.get("name", "")).strip()
            if name:
                names.append(name)
    if top_n is not None:
        names = names[:top_n]
    return " ".join(names)

def extract_director(x):
    items = safe_parse_list(x)
    for item in items:
        if isinstance(item, dict) and item.get("job") == "Director":
            return str(item.get("name", "")).strip()
    return ""

def to_num(series):
    return pd.to_numeric(series, errors="coerce")

def clean_text(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# ============================================================
# STEP 0: CHECK FILES AND COPY TO LOCAL
# ============================================================
print("0. Checking raw files in Google Drive...")

require_file(MOVIES_FILE)
require_file(KEYWORDS_FILE)
require_file(CREDITS_FILE)

copy_to_local(MOVIES_FILE, MOVIES_LOCAL)
copy_to_local(KEYWORDS_FILE, KEYWORDS_LOCAL)
copy_to_local(CREDITS_FILE, CREDITS_LOCAL)

if os.path.exists(LINKS_FILE):
    copy_to_local(LINKS_FILE, LINKS_LOCAL)
    has_links = True
else:
    has_links = False
    print("   links.csv not found. Continuing without movieId bridge.")

print("   Raw files ready.")

# ============================================================
# STEP 1: LOAD AND CLEAN movies_metadata.csv
# ============================================================
print("\n1. Loading and cleaning movies_metadata.csv ...")

meta_cols = [
    "id", "imdb_id", "title", "original_title", "overview",
    "genres", "original_language", "release_date",
    "budget", "revenue", "runtime",
    "popularity", "vote_average", "vote_count"
]

movies = pd.read_csv(
    MOVIES_LOCAL,
    usecols=lambda c: c in meta_cols,
    low_memory=False
)

raw_movies_rows = len(movies)

# ID cleaning
movies["id"] = to_num(movies["id"])
movies = movies.dropna(subset=["id"]).copy()
movies["id"] = movies["id"].astype("int64")
movies = movies.drop_duplicates(subset="id").copy()

# Numeric cleaning
for col in ["budget", "revenue", "runtime", "popularity", "vote_average", "vote_count"]:
    movies[col] = to_num(movies[col])

# Text cleaning
movies["title"] = clean_text(movies["title"])
movies["original_title"] = clean_text(movies["original_title"])
movies["overview"] = clean_text(movies["overview"])
movies["imdb_id"] = clean_text(movies["imdb_id"])
movies["original_language"] = clean_text(movies["original_language"])

# Date cleaning
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["release_year"] = movies["release_date"].dt.year.astype("Int64")

# Parsed genres
movies["genres_clean"] = clean_text(movies["genres"].apply(extract_names))

# Keep only readable imdb_id, title, and parsed genres
# original_title and raw genres will be dropped later

same_title_ratio = (movies["title"] == movies["original_title"]).mean()

print(f"   Raw rows: {raw_movies_rows:,}")
print(f"   After id cleaning & deduplication: {len(movies):,}")
print(f"   Share where title == original_title: {same_title_ratio:.2%}")

# ============================================================
# STEP 2: LOAD AND CLEAN keywords.csv
# ============================================================
print("\n2. Loading and cleaning keywords.csv ...")

keywords = pd.read_csv(KEYWORDS_LOCAL, low_memory=False)
raw_keywords_rows = len(keywords)

keywords["id"] = to_num(keywords["id"])
keywords = keywords.dropna(subset=["id"]).copy()
keywords["id"] = keywords["id"].astype("int64")
keywords = keywords.drop_duplicates(subset="id").copy()

keywords["keywords_clean"] = clean_text(keywords["keywords"].apply(extract_names))
keywords = keywords[["id", "keywords_clean"]]

print(f"   Raw rows: {raw_keywords_rows:,}")
print(f"   After cleaning: {len(keywords):,}")

# ============================================================
# STEP 3: LOAD AND CLEAN credits.csv
# ============================================================
print("\n3. Loading and cleaning credits.csv ...")

credits = pd.read_csv(CREDITS_LOCAL, low_memory=False)
raw_credits_rows = len(credits)

credits["id"] = to_num(credits["id"])
credits = credits.dropna(subset=["id"]).copy()
credits["id"] = credits["id"].astype("int64")
credits = credits.drop_duplicates(subset="id").copy()

credits["cast_clean"] = clean_text(credits["cast"].apply(lambda x: extract_names(x, top_n=TOP_CAST)))
credits["director_name"] = clean_text(credits["crew"].apply(extract_director))

credits = credits[["id", "cast_clean", "director_name"]]

print(f"   Raw rows: {raw_credits_rows:,}")
print(f"   After cleaning: {len(credits):,}")

# ============================================================
# STEP 4: OPTIONAL links.csv FOR movieId
# ============================================================
print("\n4. Loading and cleaning links.csv ...")

links = None
if has_links:
    links = pd.read_csv(LINKS_LOCAL, low_memory=False)
    raw_links_rows = len(links)

    if "movieId" in links.columns:
        links["movieId"] = to_num(links["movieId"])
    if "tmdbId" in links.columns:
        links["tmdbId"] = to_num(links["tmdbId"])

    links = links.dropna(subset=["tmdbId"]).copy()
    links["tmdbId"] = links["tmdbId"].astype("int64")

    if "movieId" in links.columns:
        links["movieId"] = links["movieId"].astype("Int64")

    links = links.rename(columns={"tmdbId": "id"})
    keep_cols = [c for c in ["id", "movieId"] if c in links.columns]
    links = links[keep_cols].drop_duplicates(subset="id")

    print(f"   Raw rows: {raw_links_rows:,}")
    print(f"   After cleaning: {len(links):,}")
else:
    print("   links.csv not available. movieId will be missing.")

# ============================================================
# STEP 5: MERGE TABLES
# ============================================================
print("\n5. Merging all cleaned tables ...")

master = movies.merge(keywords, on="id", how="left")
master = master.merge(credits, on="id", how="left")

if links is not None:
    master = master.merge(links, on="id", how="left")
else:
    master["movieId"] = pd.Series([pd.NA] * len(master), dtype="Int64")

# Fill merged text fields
master["keywords_clean"] = clean_text(master["keywords_clean"])
master["cast_clean"] = clean_text(master["cast_clean"])
master["director_name"] = clean_text(master["director_name"])

# ============================================================
# STEP 6: PROPER FOUNDATION CLEANING
# ============================================================
print("\n6. Applying final foundation cleaning ...")

# one row per movie
master = master.drop_duplicates(subset="id").copy()

# remove movies with no title
master = master[master["title"].str.len() > 0].copy()

# keep overview as clean string
master["overview"] = clean_text(master["overview"])

# financial cleaning
master["budget"] = master["budget"].where(master["budget"] > 0, np.nan)
master["revenue"] = master["revenue"].where(master["revenue"] >= 0, np.nan)
master["runtime"] = master["runtime"].where(master["runtime"] > 0, np.nan)
master["vote_count"] = master["vote_count"].where(master["vote_count"] >= 0, np.nan)
master["vote_average"] = master["vote_average"].where(
    (master["vote_average"] >= 0) & (master["vote_average"] <= 10), np.nan
)
master["popularity"] = master["popularity"].where(master["popularity"] >= 0, np.nan)

# ROI
master["roi_raw"] = np.where(
    master["budget"].notna() & master["revenue"].notna() & (master["budget"] > 0),
    master["revenue"] / master["budget"],
    np.nan
)

# ============================================================
# STEP 7: BUILD THE THREE TEXT COLUMNS
# ============================================================
print("\n7. Building text columns ...")

master["text_overview_only"] = master["overview"]

master["text_semantic_enriched"] = (
    master["title"].fillna("").astype(str).str.strip() + " " +
    master["keywords_clean"].fillna("").astype(str).str.strip() + " " +
    master["overview"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

master["text_semantic_full"] = (
    master["title"].fillna("").astype(str).str.strip() + " " +
    master["keywords_clean"].fillna("").astype(str).str.strip() + " " +
    master["cast_clean"].fillna("").astype(str).str.strip() + " " +
    master["overview"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

# ============================================================
# STEP 8: FINAL COLUMN SELECTION
# ============================================================
print("\n8. Selecting final canonical schema ...")

final_cols = [
    "id",
    "movieId",
    "imdb_id",
    "title",
    "overview",
    "keywords_clean",
    "cast_clean",
    "director_name",
    "genres_clean",
    "original_language",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "roi_raw",
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "text_overview_only",
    "text_semantic_enriched",
    "text_semantic_full"
]

for col in final_cols:
    if col not in master.columns:
        if col == "movieId":
            master[col] = pd.Series([pd.NA] * len(master), dtype="Int64")
        else:
            master[col] = np.nan

master = master[final_cols].copy()
master = master.sort_values("id").reset_index(drop=True)

# ============================================================
# STEP 9: SAVE FILE
# ============================================================
print("\n9. Saving thesis_master_base.csv ...")

master.to_csv(OUTPUT_LOCAL, index=False)
shutil.copy2(OUTPUT_LOCAL, OUTPUT_DRIVE)

# ============================================================
# STEP 10: SUMMARY
# ============================================================
print("\n✅ DONE")
print(f"Saved local file : {OUTPUT_LOCAL}")
print(f"Saved drive file : {OUTPUT_DRIVE}")
print(f"Final shape      : {master.shape[0]:,} rows x {master.shape[1]} columns")

print("\nFinal columns:")
for c in master.columns:
    print("-", c)

print("\nMissing values (%):")
missing_pct = (master.isna().mean() * 100).round(2)
print(missing_pct)

print("\nSample rows:")
display(master.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
0. Checking raw files in Google Drive...
   Raw files ready.

1. Loading and cleaning movies_metadata.csv ...
   Raw rows: 45,466
   After id cleaning & deduplication: 45,433
   Share where title == original_title: 74.93%

2. Loading and cleaning keywords.csv ...
   Raw rows: 46,419
   After cleaning: 45,432

3. Loading and cleaning credits.csv ...
   Raw rows: 45,476
   After cleaning: 45,432

4. Loading and cleaning links.csv ...
   Raw rows: 45,843
   After cleaning: 45,594

5. Merging all cleaned tables ...

6. Applying final foundation cleaning ...

7. Building text columns ...

8. Selecting final canonical schema ...

9. Saving thesis_master_base.csv ...

✅ DONE
Saved local file : /content/thesis_movie_project/processed/thesis_master_base.csv
Saved drive file : /content/drive/MyDrive/thesis_movie_project/processed/thesis_master_base.csv
Final shape     

,id,movieId,imdb_id,title,overview,keywords_clean,cast_clean,director_name,genres_clean,original_language,...,budget,revenue,roi_raw,runtime,popularity,vote_average,vote_count,text_overview_only,text_semantic_enriched,text_semantic_full
0,2,4470,tt0094675,Ariel,Taisto Kasurinen is a Finnish coal miner whose...,underdog prison factory worker prisoner helsin...,Turo Pajala Susanna Haavisto Matti Pellonpää E...,Aki Kaurismäki,Drama Crime,fi,...,NaN,0.0,NaN,69.0,3.860491,7.1,44.0,Taisto Kasurinen is a Finnish coal miner whose...,Ariel underdog prison factory worker prisoner ...,Ariel underdog prison factory worker prisoner ...
1,3,61724,tt0092149,Shadows in Paradise,"An episode in the life of Nikander, a garbage ...",salesclerk helsinki garbage independent film,Matti Pellonpää Kati Outinen Sakari Kuosmanen ...,Aki Kaurismäki,Drama Comedy,fi,...,NaN,0.0,NaN,76.0,2.292110,7.1,35.0,"An episode in the life of Nikander, a garbage ...",Shadows in Paradise salesclerk helsinki garbag...,Shadows in Paradise salesclerk helsinki garbag...
2,5,18,tt0113101,Four Rooms,It's Ted the Bellhop's first night on the job....,hotel new year's eve witch bet hotel room sper...,Tim Roth Antonio Banderas Jennifer Beals Madon...,Allison Anders,Crime Comedy,en,...,4000000.0,4300000.0,1.075000,98.0,9.026586,6.5,539.0,It's Ted the Bellhop's first night on the job....,Four Rooms hotel new year's eve witch bet hote...,Four Rooms hotel new year's eve witch bet hote...
3,6,479,tt0107286,Judgment Night,"While racing to a boxing match, Frank, Mike, J...",chicago drug dealer boxing match escape one night,Emilio Estevez Cuba Gooding Jr. Denis Leary Je...,Stephen Hopkins,Action Thriller Crime,en,...,NaN,12136938.0,NaN,110.0,5.538671,6.4,79.0,"While racing to a boxing match, Frank, Mike, J...",Judgment Night chicago drug dealer boxing matc...,Judgment Night chicago drug dealer boxing matc...
4,11,260,tt0076759,Star Wars,Princess Leia is captured and held hostage by ...,android galaxy hermit death star lightsaber je...,Mark Hamill Harrison Ford Carrie Fisher Peter ...,George Lucas,Adventure Action Science Fiction,en,...,11000000.0,775398007.0,70.490728,121.0,42.149697,8.1,6778.0,Princess Leia is captured and held hostage by ...,Star Wars android galaxy hermit death star lig...,Star Wars android galaxy hermit death star lig...
5,12,6377,tt0266543,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",father son relationship harbor underwater fish...,Albert Brooks Ellen DeGeneres Alexander Gould ...,Andrew Stanton,Animation Family,en,...,94000000.0,940335536.0,10.003570,100.0,25.497794,7.6,6292.0,"Nemo, an adventurous young clownfish, is unexp...",Finding Nemo father son relationship harbor un...,Finding Nemo father son relationship harbor un...
6,13,356,tt0109830,Forrest Gump,A man with a low IQ has accomplished great thi...,vietnam veteran hippie mentally disabled runni...,Tom Hanks Robin Wright Gary Sinise Mykelti Wil...,Robert Zemeckis,Comedy Drama Romance,en,...,55000000.0,677945399.0,12.326280,142.0,48.307194,8.2,8147.0,A man with a low IQ has accomplished great thi...,Forrest Gump vietnam veteran hippie mentally d...,Forrest Gump vietnam veteran hippie mentally d...
7,14,2858,tt0169547,American Beauty,"Lester Burnham, a depressed suburban father in...",male nudity female nudity adultery parent chil...,Kevin Spacey Annette Bening Thora Birch Wes Be...,Sam Mendes,Drama,en,...,15000000.0,356296601.0,23.753107,122.0,20.726578,7.9,3438.0,"Lester Burnham, a depressed suburban father in...",American Beauty male nudity female nudity adul...,American Beauty male nudity female nudity adul...
8,15,923,tt0033467,Citizen Kane,"Newspaper magnate, Charles Foster Kane is take...",journalist newspaper florida capitalist banker...,Orson Welles Joseph Cotten Dorothy Comingore R...,Orson Welles,Mystery Drama,en,...,839727.0,23217674.0,27.649074,119.0,15.811921,8.0,1244.0,"Newspaper magnate, Charles Foster Kane is take...",Citizen Kane journalist newspaper florida capi...,Citi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

BASE_DIR = "/content/drive/MyDrive/thesis_movie_project"
RAW_DIR = f"{BASE_DIR}/raw"
PROCESSED_DIR = f"{BASE_DIR}/processed"

MASTER_PATH = f"{PROCESSED_DIR}/thesis_master_final.csv"
LINKS_PATH = f"{RAW_DIR}/links.csv"
RATINGS_PATH = f"{RAW_DIR}/ratings.csv"
OUTPUT_PATH = f"{PROCESSED_DIR}/thesis_master_final_rich.csv"

# ------------------------------
# 1. Load master + links
# ------------------------------
master = pd.read_csv(MASTER_PATH, low_memory=False)
links = pd.read_csv(LINKS_PATH, low_memory=False)

links["movieId"] = pd.to_numeric(links["movieId"], errors="coerce")
links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")

links = links.dropna(subset=["movieId", "tmdbId"]).copy()
links["movieId"] = links["movieId"].astype("int64")
links["tmdbId"] = links["tmdbId"].astype("int64")

# ------------------------------
# 2. Aggregate ratings in chunks
# ------------------------------
chunk_size = 2_000_000
agg_parts = []

for chunk in pd.read_csv(RATINGS_PATH, chunksize=chunk_size):
    chunk["movieId"] = pd.to_numeric(chunk["movieId"], errors="coerce")
    chunk["rating"] = pd.to_numeric(chunk["rating"], errors="coerce")
    chunk = chunk.dropna(subset=["movieId", "rating"]).copy()
    chunk["movieId"] = chunk["movieId"].astype("int64")

    part = chunk.groupby("movieId").agg(
        rating_sum=("rating", "sum"),
        rating_count=("rating", "size"),
        rating_sq_sum=("rating", lambda x: np.sum(np.square(x))),
        rating_median=("rating", "median")
    ).reset_index()

    agg_parts.append(part)

ratings_agg = pd.concat(agg_parts, ignore_index=True)

# Combine chunk aggregates
ratings_agg = ratings_agg.groupby("movieId").agg(
    rating_sum=("rating_sum", "sum"),
    rating_count=("rating_count", "sum"),
    rating_sq_sum=("rating_sq_sum", "sum"),
    rating_median=("rating_median", "median")
).reset_index()

ratings_agg["ml_rating_mean"] = ratings_agg["rating_sum"] / ratings_agg["rating_count"]

# variance/std from sums
ratings_agg["ml_rating_var"] = (
    ratings_agg["rating_sq_sum"] / ratings_agg["rating_count"]
    - np.square(ratings_agg["ml_rating_mean"])
)
ratings_agg["ml_rating_var"] = ratings_agg["ml_rating_var"].clip(lower=0)
ratings_agg["ml_rating_std"] = np.sqrt(ratings_agg["ml_rating_var"])

ratings_agg = ratings_agg.rename(columns={"rating_count": "ml_rating_count"})

# ------------------------------
# 3. Bayesian weighted rating
# ------------------------------
C = ratings_agg["ml_rating_mean"].mean()
m = ratings_agg["ml_rating_count"].quantile(0.75)

ratings_agg["ml_weighted_rating"] = (
    (ratings_agg["ml_rating_count"] / (ratings_agg["ml_rating_count"] + m)) * ratings_agg["ml_rating_mean"]
    + (m / (ratings_agg["ml_rating_count"] + m)) * C
)

ratings_agg = ratings_agg[[
    "movieId",
    "ml_rating_count",
    "ml_rating_mean",
    "ml_rating_std",
    "rating_median",
    "ml_weighted_rating"
]].copy()

ratings_agg = ratings_agg.rename(columns={"rating_median": "ml_rating_median"})

# ------------------------------
# 4. Merge ratings with links
# ------------------------------
ratings_linked = links.merge(ratings_agg, on="movieId", how="inner")
ratings_linked = ratings_linked.rename(columns={"tmdbId": "id"})

# ------------------------------
# 5. Merge into master
# ------------------------------
master["id"] = pd.to_numeric(master["id"], errors="coerce")
master = master.dropna(subset=["id"]).copy()
master["id"] = master["id"].astype("int64")

master_rich = master.merge(
    ratings_linked[[
        "id", "movieId", "ml_rating_count", "ml_rating_mean",
        "ml_rating_std", "ml_rating_median", "ml_weighted_rating"
    ]],
    on="id",
    how="left",
    suffixes=("", "_links")
)

# If movieId is missing in master but present from links, backfill it
if "movieId_links" in master_rich.columns:
    master_rich["movieId"] = master_rich["movieId"].fillna(master_rich["movieId_links"])
    master_rich = master_rich.drop(columns=["movieId_links"])

# ------------------------------
# 6. Save
# ------------------------------
master_rich.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", master_rich.shape)

print("\nMissing values in new MovieLens columns (%):")
new_cols = ["ml_rating_count", "ml_rating_mean", "ml_rating_std", "ml_rating_median", "ml_weighted_rating"]
print((master_rich[new_cols].isna().mean() * 100).round(2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/MyDrive/thesis_movie_project/processed/thesis_master_final_rich.csv
Shape: (89599, 27)

Missing values in new MovieLens columns (%):
ml_rating_count       50.07
ml_rating_mean        50.07
ml_rating_std         50.07
ml_rating_median      50.07
ml_weighted_rating    50.07
dtype: float64
